# Day 51 — Two-agent chat: author + critic

The reflection idea from Day 21, now as two agents. The **author** drafts; the **critic**
approves or asks for one fix. They alternate until the critic says `APPROVE`.

> Install: `pip install "autogen-agentchat" "autogen-ext[openai]"` and set `OPENAI_API_KEY`. For a local model, pass `base_url=` + `model_info=` to the client.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run. Solution below.

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
author = AssistantAgent("author", model_client=model_client,
    system_message="Write or revise a short slogan based on feedback.")
critic = AssistantAgent("critic", model_client=model_client,
    system_message="If the slogan is great, reply exactly 'APPROVE'. Otherwise give ONE concrete fix.")

# TODO: build a RoundRobinGroupChat([author, critic]) that stops on 'APPROVE' OR after 6 messages,
#       run it on "a slogan for loose-leaf tea", and print each message.


## 🔒 Solution

Verified against autogen-agentchat 0.7.5.

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
author = AssistantAgent("author", model_client=model_client,
    system_message="Write or revise a short slogan based on feedback.")
critic = AssistantAgent("critic", model_client=model_client,
    system_message="If the slogan is great, reply exactly 'APPROVE'. Otherwise give ONE concrete fix.")

team = RoundRobinGroupChat(
    [author, critic],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(6),
)
result = await team.run(task="Write a slogan for loose-leaf tea.")
for m in result.messages:
    print(f"{m.source:8}: {m.content}")
await model_client.close()